First job we will simply examine the data

In [1]:
from pathlib import Path
import os
import sys
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / 'project_code_2026_publicshare',
                   parent / 'project_2026/project_code_2026_publicshare',
                   parent / 'private_github/project_2026/project_code_2026_publicshare')
    if (folder / 'pricing_helpers.py').is_file()
), None)
if project_folder is None:
    raise FileNotFoundError('Open the project folder as your notebook workspace.')
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

In [2]:
import statsmodels.formula.api as smf

def fit_logistic_regression_demand_with_covariates(df):
    # Make a copy so we don't modify original df outside the function
    data = df.copy()

    data["item_bought"] = data["item_bought"].astype(int)

        
    formula = 'item_bought ~ Covariate1 * Covariate2 * Covariate3 + price_item'
    
    fitted_model = smf.logit(formula=formula, data=data).fit()

    return fitted_model

In [3]:
data = pd.read_csv(project_folder / 'data/train_prices_decisions_2026.csv')
train, test = train_test_split(data, test_size=.2, random_state=0)
model = fit_logistic_regression_demand_with_covariates(train)
model.summary()

Optimization terminated successfully.
         Current function value: 0.349035
         Iterations 9


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            item_bought   No. Observations:                40000
Model:                          Logit   Df Residuals:                    39991
Method:                           MLE   Df Model:                            8
Date:                Mon, 02 Nov 2026   Pseudo R-squ.:                  0.4874
Time:                        15:31:38   Log-Likelihood:                -13961.
converged:                       True   LL-Null:                       -27235.
Covariance Type:            nonrobust   LLR p-value:                     0.000
====================================================================================================
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            2.1922      0.079     27.844      0.000       2.038       2.346
Covariate1                           0.3570      0.092      3.864      0.000       0.176       0.538
Covariate2                          -0.2400      0.019    -12.652      0.000      -0.277      -0.203
Covariate1:Covariate2                0.5351      0.023     23.030      0.000       0.490       0.581
Covariate3                           0.1822      0.008     23.631      0.000       0.167       0.197
Covariate1:Covariate3               -0.0281      0.009     -3.003      0.003      -0.046      -0.010
Covariate2:Covariate3                0.0045      0.002      2.390      0.017       0.001       0.008
Covariate1:Covariate2:Covariate3    -0.0227      0.002    -10.445      0.000      -0.027      -0.018
price_item                          -0.0655      0.001    -83.063      0.000      -0.067      -0.064
====================================================================================================

Possibly complete quasi-separation: A fraction 0.18 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

In [4]:
features = ['Covariate1', 'Covariate2', 'Covariate3', 'price_item']

purchase_probability = model.predict(test[features])

print(
    'Held-out mean squared error:',
    np.mean((purchase_probability - test.item_bought.astype(int)) ** 2)
)

Held-out mean squared error: 0.08362764013382556


In [5]:
model_path = project_folder / 'agents/helper_files_here/purchase_model1.pkl'

#example on how to save a model in a pickle file
with model_path.open('wb') as file:
    pickle.dump(model, file)
    
#how to open the saved model -- you will need to do this in your agents file
with model_path.open('rb') as file:
    loaded_model = pickle.load(file)
    
np.testing.assert_allclose(loaded_model.predict(test[features]), model.predict(test[features]))
print('Saved and reloaded:', model_path)

Saved and reloaded: c:\Users\leoca\OneDrive\Desktop\project\Algorithmic_Pricing_Agent\project\agents\helper_files_here\purchase_model1.pkl
